# 12.6: A minimal judge

A pairwise judge is a prompt template plus parsing. Here is a sketch using a generic chat-completion function; any LLM API or local model can fill in `call_llm`.


In [ ]:
JUDGE_TEMPLATE = """You are an impartial evaluator. Compare two responses to the user's question.
Judge helpfulness, correctness, and adherence to the question. Do not let response length,
formatting, or the order of the responses affect your decision.

[Question]
{question}

[Response A]
{a}

[Response B]
{b}

Explain your reasoning briefly, then output exactly one final line:
"Verdict: A", "Verdict: B", or "Verdict: tie"."""

def parse_verdict(text: str) -> str:
    for line in reversed(text.strip().splitlines()):
        if line.strip().lower().startswith("verdict:"):
            return line.split(":", 1)[1].strip().lower()
    return "invalid"

def judge_pair(question, resp_1, resp_2, call_llm) -> float:
    """Returns resp_1's score in [0, 1], judging both orders to cancel position bias."""
    v1 = parse_verdict(call_llm(JUDGE_TEMPLATE.format(question=question, a=resp_1, b=resp_2)))
    v2 = parse_verdict(call_llm(JUDGE_TEMPLATE.format(question=question, a=resp_2, b=resp_1)))
    score_1 = {"a": 1.0, "tie": 0.5, "b": 0.0}.get(v1, 0.5)
    score_2 = {"b": 1.0, "tie": 0.5, "a": 0.0}.get(v2, 0.5)   # order swapped
    return (score_1 + score_2) / 2
